## Week 7 — Task 2: Feature Engineering  (Uber Rides)

We are loading a datetime-rich taxi ride dataset. We will train a baseline model, engineer 6 distinct features, and compare accuracy metrics to isolate performance changes.


In [4]:
import numpy as np
import pandas as pd

# Set random seed for reproducibility
np.random.seed(42)
n_rides = 1000

# 1. Generate fake Uber/Taxi data directly on your machine
mock_uber_data = {
    "pickup_datetime": pd.date_range(
        start="2026-01-01 00:00:00", periods=n_rides, freq="35min"
    ),
    "pickup_longitude": np.random.uniform(-74.05, -73.90, size=n_rides),
    "pickup_latitude": np.random.uniform(40.65, 40.85, size=n_rides),
    "passenger_count": np.random.choice(
        [1, 2, 5, 6], size=n_rides, p=[0.7, 0.15, 0.1, 0.05]
    ),
    "trip_distance": np.random.uniform(0.5, 15.0, size=n_rides),
    "fare_amount": np.random.uniform(5.0, 50.0, size=n_rides),
}

# 2. Convert to DataFrame
df_uber = pd.DataFrame(mock_uber_data)

# 3. Create our binary target column (Is it an expensive ride?)
median_fare = df_uber["fare_amount"].median()
df_uber["IsExpensive"] = (df_uber["fare_amount"] > median_fare).astype(int)

print("Dataset Shape:", df_uber.shape)
df_uber.head(10)


Dataset Shape: (1000, 7)


,pickup_datetime,pickup_longitude,pickup_latitude,passenger_count,trip_distance,fare_amount,IsExpensive
0,2026-01-01 00:00:00,-73.993819,40.687027,1,10.254193,30.739815,1
1,2026-01-01 00:35:00,-73.907393,40.758380,1,12.051880,41.244455,1
2,2026-01-01 01:10:00,-73.940201,40.824589,5,4.131785,39.207242,1
3,2026-01-01 01:45:00,-73.960201,40.796445,1,9.560674,11.925496,0
4,2026-01-01 02:20:00,-74.026597,40.811312,1,8.790317,11.716226,0
5,2026-01-01 02:55:00,-74.026601,40.781757,2,12.576040,17.067847,0
6,2026-01-01 03:30:00,-74.041287,40.788455,1,13.638262,21.248363,0
7,2026-01-01 04:05:00,-73.920074,40.819839,2,0.676273,23.380501,0
8,2026-01-01 04:40:00,-73.959833,40.699934,1,10.273289,35.586375,1
9,2026-01-01 05:15:00,-73.943789,40.747885,1,1.251619,7.550619,0


## Step 1: Establish a Baseline Model (Before Feature Engineering)

We will isolate a baseline feature framework utilizing only the basic raw numerical properties: `pickup_longitude`, `pickup_latitude`, and `passenger_count`.


In [5]:
# Select raw attributes for baseline evaluation
baseline_features = ["pickup_longitude", "pickup_latitude", "passenger_count"]

X_base = df_uber[baseline_features]
y_base = df_uber["IsExpensive"]

# 80/20 Baseline Split
X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
    X_base, y_base, test_size=0.2, random_state=42
)

# Train a simple Random Forest Classifier
clf_base = RandomForestClassifier(n_estimators=50, random_state=42)
clf_base.fit(X_train_b, y_train_b)

baseline_acc = accuracy_score(y_test_b, clf_base.predict(X_test_b))
print(f"Baseline Model Accuracy: {baseline_acc:.4f}")


Baseline Model Accuracy: 0.5500


## Step 2: Engineer 6 New Features

### Justification of Engineered Features:
1. **`pickup_hour`**: Extracts the exact hour. Taxi demand and pricing scale up dramatically during morning rush hours or late-night windows.
2. **`day_of_week`**: Pulls numeric calendar days (0=Monday, 6=Sunday). Differentiates standard commute schedules from recreational weekend trends.
3. **`is_weekend`**: Isolates flags for Saturday and Sunday. Travel habits change significantly when professional environments close.
4. **`coordinate_interaction`**: Multiplies longitude by latitude. This acts as a spatial crossing indicator to map out localized zone patterns.
5. **`log_trip_distance`**: Transforms highly skewed mileage counts using `np.log1p` to distribute values more uniformly for standard algorithms.
6. **`passenger_bin`**: Groups raw counts into distinct categorical brackets ("Solo", "Small Group", "Large Group") via binning boundaries.


In [6]:
# Create a copy to implement engineering operations safely
df_eng = df_uber.copy()

# 1. Extract Hour
df_eng["pickup_hour"] = df_eng["pickup_datetime"].dt.hour

# 2. Extract Day of the Week
df_eng["day_of_week"] = df_eng["pickup_datetime"].dt.dayofweek

# 3. Extract Is Weekend (5 = Saturday, 6 = Sunday)
df_eng["is_weekend"] = df_eng["day_of_week"].apply(
    lambda x: 1 if x >= 5 else 0
)

# 4. Interaction Feature (Multiplying coordinates)
df_eng["coordinate_interaction"] = (
    df_eng["pickup_longitude"] * df_eng["pickup_latitude"]
)

# 5. Log Transform on a skewed numeric column (trip_distance)
# np.log1p handles zero distances safely without returning mathematical errors
df_eng["log_trip_distance"] = np.log1p(df_eng["trip_distance"])

# 6. Binning a continuous variable (passenger_count)
bins = [-1, 1, 4, 10]
labels = [0, 1, 2]  # 0: Solo, 1: Small Group, 2: Large Group (Encoded as integers)
df_eng["passenger_bin"] = pd.cut(
    df_eng["passenger_count"], bins=bins, labels=labels
).astype(int)

# View our newly created features
new_features = [
    "pickup_hour",
    "day_of_week",
    "is_weekend",
    "coordinate_interaction",
    "log_trip_distance",
    "passenger_bin",
]
df_eng[new_features].head()


,pickup_hour,day_of_week,is_weekend,coordinate_interaction,log_trip_distance,passenger_bin
0,0,3,0,-3010.588480,2.420741,0
1,0,3,0,-3012.345617,2.568932,0
2,1,3,0,-3018.578325,1.635453,2
3,1,3,0,-3017.313280,2.357137,0
4,2,3,0,-3021.122572,2.281394,0


## Step 3: Train and Evaluate the Post-Engineering Model

We will add our 6 engineered attributes back to our original raw feature set, re-run our Random Forest classification, and calculate our accuracy improvements.


In [7]:
# Combine baseline features with the 6 newly engineered properties
final_features = baseline_features + new_features

X_eng = df_eng[final_features]
y_eng = df_eng["IsExpensive"]

# 80/20 Engineered Split
X_train_e, X_test_e, y_train_e, y_test_e = train_test_split(
    X_eng, y_eng, test_size=0.2, random_state=42
)

# Train the exact same classifier on the newly enriched dataset
clf_eng = RandomForestClassifier(n_estimators=50, random_state=42)
clf_eng.fit(X_train_e, y_train_e)

engineered_acc = accuracy_score(y_test_e, clf_eng.predict(X_test_e))
print(f"Engineered Model Accuracy: {engineered_acc:.4f}")


Engineered Model Accuracy: 0.5250


## Step 4: Final Results & Comparison Delta

### Accuracy Performance Summary:


In [8]:
# Calculate accuracy delta
accuracy_delta = engineered_acc - baseline_acc

print(f"Baseline Accuracy Score:   {baseline_acc:.4f}")
print(f"Engineered Accuracy Score: {engineered_acc:.4f}")
print(f"----------------------------------------")
print(f"Performance Delta:         {accuracy_delta:+.4f} ({accuracy_delta*100:+.2f}%)")


Baseline Accuracy Score:   0.5500
Engineered Accuracy Score: 0.5250
----------------------------------------
Performance Delta:         -0.0250 (-2.50%)


### Task 2 Discussion & Analysis of Results

* **Accuracy Delta Observed**: -0.0250 (-2.50%)
* **Why the performance dropped**: The dataset utilized for this task was generated synthetically using purely random distributions (`np.random`). Because there is no mathematical correlation between the engineered datetime properties and the target `IsExpensive` variable, adding these features introduced "noise" rather than "signal". The Random Forest classifier overfit to these irrelevant patterns, resulting in a slight drop in generalization accuracy on the testing set.
* **Practical Takeaway**: Feature engineering is highly effective on real-world datasets where chronological patterns exist (e.g., higher taxi fares during rush hour). However, adding features without a genuine underlying relationship to the target variable will degrade model performance.
